# Drishti — End-to-End App Test (on Colab)

Runs the **real `app/` code** — router, engines, guardrail, translation, speech — against
real photos, on a Colab T4. This is the Phase-1 exit criterion, executed without installing
anything locally.

Everything in `app/` is unit-tested with fakes (117 tests). What has never happened is a
real model loading through it. That is what this notebook checks.

### Why the repo has to be uploaded

The Colab VS Code extension runs *cells* on a Colab machine; it does not copy your project
there. `app/` therefore does not exist on the runtime until we put it there. Cell 1 handles
that.

### Order matters

OCR (PaddlePaddle) and everything else (PyTorch) each bundle an OpenMP runtime, and
co-loading them killed the kernel during the OCR spike with no traceback (`DEC-006`).
`KMP_DUPLICATE_LIB_OK` is set before any import as a mitigation, and the modes are run
**OCR first, VLM last** so that if the process does die you still have the earlier results.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import platform, os
print(platform.system(), platform.release())
print('hostname:', platform.node())
print('cwd:', os.getcwd())
!nvidia-smi --query-gpu=name --format=csv,noheader


## 1. Get the project onto the runtime

The Colab VS Code extension runs *cells* on Colab hardware but leaves your files on the
local disk, so `app/` does not exist on the runtime until we put it there.

> **`google.colab.files.upload()` does not work from VS Code.** It is a browser widget: the
> HTML renders, the JavaScript bridge that picks the file never loads, and the cell hangs
> until you interrupt it. Use one of the two paths below instead.

### Path A — git clone (recommended)

Push the project to GitHub once, then set `REPO_URL` below. Every later run is a single cell
that always pulls current code, and the repo ends up backed up and shareable with your guide.

```powershell
git remote add origin https://github.com/<you>/drishti.git
git push -u origin main
```

### Path B — run this notebook in the Colab browser

Open [colab.research.google.com](https://colab.research.google.com), upload this notebook,
and `files.upload()` behaves normally. Zero setup, but you lose the VS Code editor.

Sample photos are committed under `data/samples/`, so **no image upload is needed either
way** — that failure mode is gone entirely.

In [ ]:
import os

# Set before any framework import -- see DEC-006.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = ''          # <-- Path A: 'https://github.com/<you>/drishti.git'
PROJECT = Path('/content/drishti')


def _find_project_root(start: Path):
    """Locate the folder holding app/router.py, however the archive nested it."""
    if (start / 'app' / 'router.py').exists():
        return start
    for marker in start.glob('*/app/router.py'):
        return marker.parents[1]
    return None


if not (PROJECT / 'app' / 'router.py').exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/_clone'],
                       check=True)
        found = _find_project_root(Path('/content/_clone'))
        if found is None:
            raise SystemExit('app/router.py not found in the cloned repo.')
        shutil.move(str(found), str(PROJECT))
    else:
        # Path B only -- this widget works in the Colab browser, never from VS Code.
        try:
            from google.colab import files
        except ImportError:
            raise SystemExit('Not running on Colab. Set REPO_URL above.')
        print('Upload drishti.zip  (Colab browser only; from VS Code set REPO_URL instead)')
        up = files.upload()
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall('/content/_unpack')
        found = _find_project_root(Path('/content/_unpack'))
        if found is None:
            raise SystemExit('app/router.py not in the archive -- did you zip the drishti '
                             'folder itself?')
        shutil.move(str(found), str(PROJECT))

if not (PROJECT / 'app' / 'router.py').exists():
    raise SystemExit(f'{PROJECT}/app/router.py missing -- project not set up.')

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('project root :', PROJECT)
print('modes        :', sorted(p.stem for p in (PROJECT / 'app' / 'modes').glob('[!_]*.py')))
print('sample images:', sorted(p.name for p in (PROJECT / 'data' / 'samples').glob('*.jpg')))

In [ ]:
# The suite needs no models, so a pass here proves the upload is complete and importable
# before we spend minutes downloading weights.
!python -m unittest discover -s tests -t . 2>&1 | tail -4

## 2. Install engines

Weights are **not** downloaded here — every engine loads lazily on first use, so each mode
below pays only for what it needs.

In [ ]:
%pip install -q paddlepaddle paddleocr IndicTransToolkit
print('installed')

## 3. Choose test photos

Committed fixtures in `data/samples/` are used by default, so nothing needs uploading.

`strip_paracip.jpg` is the read that produced 55 OCR lines including the drug name,
`EXP.OCT.2026` and `Rs.10.30`. `strip_partial.jpg` is the same strip framed badly — only
3 lines — kept as the negative case.

To test **Devanagari Read mode**, drop a photo containing Marathi or Hindi text into
`data/samples/` and set `DEVANAGARI` below. That is still the one unverified claim in the
project: the code path is confirmed, the model has never seen actual Devanagari.

In [ ]:
SAMPLES = PROJECT / 'data' / 'samples'
photos = sorted(SAMPLES.glob('*.jpg'))

for i, p in enumerate(photos):
    print(f'  [{i}] {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

STRIP = SAMPLES / 'strip_paracip.jpg'      # the good read
DEVANAGARI = None                          # <-- set to a Marathi/Hindi photo to test §6

if not STRIP.exists():
    raise SystemExit(f'{STRIP} missing -- is data/samples/ present in the project?')

print('\nstrip     :', STRIP.name)
print('devanagari:', DEVANAGARI.name if DEVANAGARI else '(none set -- §6 will skip)')

## 4. Medicine mode — the guardrail, end to end

OCR reads the strip, the drug name is matched against the verified database, expiry and MRP
are parsed. If OCR cannot produce a verified name the mode **declines** rather than guessing
(`DEC-007`).

In [ ]:
import time

from app.drug_db import DrugDatabase
from app.engines.paddle_ocr import PaddleOCREngine
from app.modes.medicine import run as run_medicine

ocr = PaddleOCREngine(lang='en')

t0 = time.time()
result = run_medicine(STRIP, ocr, DrugDatabase.from_file())
elapsed = time.time() - t0

print(f'--- medicine mode  ({elapsed:.1f}s) ---')
print('verified :', result.ok)
print('drug     :', result.drug_name)
print('expiry   :', result.expiry_raw, '| expired:', result.expired)
print('MRP      :', result.mrp)
print('\nSPOKEN   :', result.message_en)

if not result.ok:
    print('\nDeclined. Either OCR missed the name, or it is absent from')
    print('data/drug_names_seed.txt -- a 30-entry placeholder, not a real drug database.')

## 5. Marathi output and speech — the Phase-1 exit criterion

In [ ]:
from IPython.display import Audio, display

from app.engines.indictrans import IndicTrans2Translator
from app.engines.mms_tts import MMSTTSEngine
from app.speech import deliver

translator = IndicTrans2Translator()
tts = MMSTTSEngine(out_dir=Path('/content/audio'))

for lang in ('mr', 'hi'):
    t0 = time.time()
    spoken = deliver(result.message_en, lang=lang, translator=translator, tts=tts, speak=True)
    print(f'--- {lang} ({time.time()-t0:.1f}s) ---')
    print(spoken.text_out)
    display(Audio(str(spoken.audio_path)))

## 6. Read mode — Devanagari

`lang='mr'` resolves to `devanagari_PP-OCRv5_mobile_rec`. The code path is confirmed; what
has never been tested is the model against actual Devanagari text.

In [ ]:
from app.modes.read import run as run_read

if DEVANAGARI is None:
    print('No second photo uploaded -- skipping. Read mode in Marathi stays unverified.')
else:
    t0 = time.time()
    text = run_read(DEVANAGARI, PaddleOCREngine(lang='mr'))
    print(f'--- read mode, devanagari ({time.time()-t0:.1f}s) ---')
    print(text)

## 7. Scene mode — the VLM

Last on purpose. This loads PyTorch beside PaddlePaddle, which is the combination that can
abort the process; running it last means a crash costs you nothing already measured.

**If the kernel dies here, that is the documented OpenMP collision, not your mistake.**
Restart, skip to this cell, and it will run on its own.

In [ ]:
from app.engines.smolvlm import SmolVLMEngine
from app.modes.ask import run as run_ask
from app.modes.scene import run as run_scene

vlm = SmolVLMEngine()

t0 = time.time()
print('--- scene mode ---')
print(run_scene(STRIP, vlm), f'({time.time()-t0:.1f}s)')

t0 = time.time()
print('\n--- ask mode ---')
print(run_ask(STRIP, vlm, 'what is written on this?'), f'({time.time()-t0:.1f}s)')

## 8. Findings — fill in, then update `docs/BUILD_PLAN.md`

| Check | Result | Latency |
|---|---|---|
| Medicine: drug name verified | | s |
| Medicine: expiry parsed | | |
| Medicine: MRP parsed | | |
| Marathi translation readable | | s |
| Marathi speech intelligible | | s |
| Hindi speech intelligible | | s |
| Devanagari Read mode | | s |
| Scene mode | | s |
| Kernel survived OCR + VLM together | | |

**Phase 1 is complete when** the medicine row is verified and Marathi audio plays. Tick
those boxes in the build plan and record the latencies against the <8 s target (RISK-1).

Ask a Marathi speaker whether the translation and the synthesized voice are actually
understandable — accuracy metrics do not capture intelligibility, and this is the first time
a human can judge the output.